In [5]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import Dense, LeakyReLU, Input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam

import matplotlib.pyplot as plt


In [6]:
LATENT_DIM = 30
DATA_DIM = 30

In [ ]:
# Bouw de generator en de discriminator modellen
def build_generator():
    model = Sequential([
        Input(shape=(LATENT_DIM,)),
        Dense(64, activation='relu'),
        Dense(128, activation='relu'),
        Dense(256, activation='relu'),
        Dense(DATA_DIM, activation='sigmoid')
    ])
    return model


def build_discriminator():
    model = Sequential([
        Input(shape=(DATA_DIM,)),
        Dense(256, activation=None),
        LeakyReLU(negative_slope=0.2),
        Dense(128, activation=None),
        LeakyReLU(negative_slope=0.2),
        Dense(1, activation='sigmoid')  # binary for nep of echt
    ])
    return model
    
    
generator = build_generator()
discriminator = build_discriminator()

generator.summary()
discriminator.summary()

# Compileren
generator.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5))
discriminator.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5))

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_28 (Dense)                │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 30)             │         7,710 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,038 (199.37 KB)

 Trainable params: 51,038 (199.37 KB)

 Non-trainable params: 0 (0.00 B)

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_32 (Dense)                │ (None, 256)            │         7,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_8 (LeakyReLU)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_9 (LeakyReLU)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,961 (160.00 KB)

 Trainable params: 40,961 (160.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Create the GAN model
discriminator.trainable = False
z = Input(shape=[LATENT_DIM,])
generated_data = generator(z)
validity = discriminator(generated_data)

gan = Model(z, validity)
gan.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5))


In [ ]:
# Define the train loop for a GAN
def train_gan(epochs=5000, batch_size=64):
    normal_data = np.random.normal(size=(10000, DATA_DIM))
    half_batch = batch_size // 2
    
    tf.keras.utils.disable_interactive_logging()

    
    for epoch in range(epochs):
        idx = np.random.randint(0, normal_data.shape[0], half_batch)
        real_data = normal_data[idx]
        
        noise = np.random.normal(size=(half_batch, LATENT_DIM))
        fake_samples = generator.predict(noise)
        
        real_labels = np.ones((half_batch, 1))
        fake_labels = np.zeros((half_batch, 1))
        
        d_loss_real = discriminator.train_on_batch(real_data, real_labels)
        d_loss_fake = discriminator.train_on_batch(fake_samples, fake_labels)
        d_loss = .5 * np.add(d_loss_real, d_loss_fake)
        
        noise = np.random.normal(0, 1, [batch_size, LATENT_DIM])
        valid_labels = np.ones((batch_size, 1))
        
        g_loss = gan.train_on_batch(noise, valid_labels)
        
        if epoch % 1000 == 0:
            print(f"Epoch {epoch+1}/{epochs} - D loss: {d_loss} - G loss: {g_loss}")
    
        
train_gan(5000, 64)

In [21]:
def anomaly_score(sample):
    sample = sample.reshape(1, -1)
    
    reconstructed_sample = generator.predict(sample)[0]
    anomaly_score = np.linalg.norm(sample - reconstructed_sample)
    
    return anomaly_score

In [ ]:
# Normale sample die binnen de distributie valt
real_transaction = np.random.normal(size=(1, DATA_DIM))

# Normale sample die buiten de distributie valt (fraude)
fraud_transaction = np.random.uniform(low=-4, high=4, size=(1, DATA_DIM))

normal_score = anomaly_score(real_transaction)
fraud_score = anomaly_score(fraud_transaction)

# Threshhold om te bepalen wat fraude is of niet (kan je natuurlijk aanpassen of dynamisch maken)
threshold = np.percentile([normal_score], 95)
normal_is_anomalous = normal_score > threshold
fraud_is_anomalous = fraud_score > threshold

print(f"Normal transaction anomaly score: {normal_score}, Classified as anomalous: {normal_is_anomalous}")
print(f"Fraud transaction anomaly score: {fraud_score}, Classified as anomalous: {fraud_is_anomalous}")

Normal transaction anomaly score: 6.698675395526148, Classified as anomalous: False
Fraud transaction anomaly score: 11.632088139898363, Classified as anomalous: True
